# Using a custom spectrum in SYOTools

In [ ]:
from syotools.models import Camera, Telescope, Source, SourcePhotometricExposure
import numpy as np
import astropy.units as u 

# The Basics of running an imaging calculation

We will use one of SYOTools' built-in templates, the Orion Nebula

In [ ]:
# create a Telescope
tel = Telescope()
tel.set_from_hwome("EAC5")
# Select an Instrument
#print(tel.instruments)
inst = tel.instruments["HRI_S.HRI_S_NIR_Imager"]

# Create a Source
source = Source() 
redshift = 0. # changes to these are not implemented yet 
extinction = 0.
magnitude = 25.0
template = "Orion Nebula"
# Configure the Source
source.set_sed(template, magnitude, redshift, extinction, bandpass="johnson,v")

# Make an Exposure
exp = SourcePhotometricExposure()
# Add the Source to the Exposure
exp.source = source
# Add the Exposure to the Instrument
inst.add_exposure(exp)

#Configure the Exposure
exp.exptime = 60 * u.s
exp.unknown = "snr"

print("Initial Template:", template)
print("SNR:", exp.snr)

But what's available in the default SYOTools spectra library? Let's look.

The library, and the functions that create it, can be loaded separately. The library is a dictionary of Synphot SourceSpectrum objects, whose input files are stored in the SYOTools repo (under data/)

In [1]:
from syotools.spectra.spec_defaults import syn_spectra_library

print(syn_spectra_library.keys())

  points: [1136.3 1142.3 1142.4 1142.5 1152.7]
  lookup_table: [-9.31015728e-07 -2.34371190e-07 -2.17769838e-06 -1.89383420e-06
 -1.86404638e-07] [synphot.models]
  points: [1170.     1170.5    1171.3    ... 2241.1    2241.2001 2241.3   ]
  lookup_table: [-3.99896421e-07 -3.16809265e-06 -1.49050587e-06 ... -8.56435582e-07
 -9.55618745e-07 -4.15987490e-08] [synphot.models]
  points: [1180.5 1181.5 1241.5 1244.5 1248.5 1249.5 1262.5 1290.5 1307.5 1310.5
 1312.5 1315.5 1316.5 1317.5 1318.5 1321.5 1330.5 1333.5 1341.5 1342.5
 1343.5 1345.5 1346.5 1348.5 1352.5 1370.5 1372.5 1379.5 1380.5 1384.5
 1386.5 1387.5 1388.5 1389.5 1391.5 1395.5 1399.5 1400.5 1426.5 1427.5
 1428.5 1429.5 1434.5 1447.5 1448.5 1449.5 1450.5 1454.5 1455.5 1456.5
 1461.5 1466.5 1474.5 1475.5 1482.5 1493.5 1498.5 1505.5 1519.5 1539.5
 1558.5 1563.5 1567.5 1569.5 1570.5 1575.5 1577.5 1579.5 1581.5 1582.5
 1584.5 1589.5 1590.5 1591.5 1592.5 1594.5 1595.5 1596.5 1599.5 1600.5
 1601.5 1605.5 1606.5 1607.5 1608.5 1609.5 1612

dict_keys(['Classical T-Tauri Star', 'M1 Dwarf', 'M Dwarf', 'G Dwarf', 'O5V Star', 'G2V Star', 'B5V Star', 'M2V Star', 'G191B2B (WD)', 'GD71 (WD)', 'GD153 (WD)', '10 Myr Starburst', 'QSO', 'Seyfert 1', 'Seyfert 2', 'Liner', 'Orion Nebula', 'Starburst, No Dust', 'Starburst, E(B-V) = 0.6', 'Elliptical Galaxy', 'Sbc Galaxy', 'NGC 1068', 'Galaxy with f_esc, HI=1, HeI=1', 'Galaxy with f_esc, HI=0.001, HeI=1', 'Flat (AB)', 'Flat in F_lambda', 'Blackbody (5000K)', 'Blackbody (100,000K)'])


# Add a new spectrum to the library.

We will use the same functions SYOTools uses to build the library in the first place.

In [ ]:
import os
from syotools.spectra.utils import load_synfits, load_fesc, load_txtfile

## Example 1: A FITS file

In [ ]:
# FITS File from the STScI TRDS Reference Atlas collection: https://archive.stsci.edu/hlsp/reference-atlases
data_path = os.path.abspath(os.path.join('..','common', 'star_galaxy'))

spectrum = {'desc': '18 Sco',
'file': [data_path, '18sco_stis_006.fits'],
'band': 'johnson,v'}

syn_spectra_library[spectrum["desc"]] = load_txtfile(spectrum)

## Example 2: A text file

In [ ]:
data_path = os.path.abspath(os.path.join('..','common', 'star_galaxy'))

spectrum = {'desc': 'A0V Star',
'file': [data_path, 'pickles_uk_9.ascii'],
'band': 'galex,fuv'}

syn_spectra_library[spectrum["desc"]] = load_txtfile(spectrum)

## Example 3: An analytic spectrum

In [ ]:
import synphot as syn
import stsynphot as stsyn
import numpy as np

# wavelength in Angstroms
wave = np.arange(100, 30000, 300) << u.AA

# Make an 8,000K blackbody
bb = syn.spectrum.SourceSpectrum(syn.models.BlackBody1D, temperature=8000)
# normalize it
bb = bb.normalize(30.0 * u.ABmag, band=stsyn.band('galex,fuv'))
# This library is built on the idea of storing flux and wavelength arrays, so we need to make this an empirical spectrum.
bb = syn.spectrum.SourceSpectrum(syn.models.Empirical1D, points=wave, lookup_table=bb(wave))

# Store the new source template in the array
syn_spectra_library['Blackbody (8,000K)'] = bb

# The Library, Revisited

Our new spectra are in there now.

In [ ]:
print(syn_spectra_library.keys())

# Use the new spectra in calculations

In [ ]:
template = "18 Sco"

source.set_sed(template, magnitude, redshift, extinction, bandpass="johnson,v")
snr = exp.calculate_snr()
print("SNR:", template, exp.snr)

template = "A0V Star"

source.set_sed(template, magnitude, redshift, extinction, bandpass="johnson,v")
snr = exp.calculate_snr()
print("SNR:", template, exp.snr)

template = "Blackbody (8,000K)"

source.set_sed(template, magnitude, redshift, extinction, bandpass="johnson,v")
snr = exp.calculate_snr()
print("SNR:", template, exp.snr)